In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%pip install great_expectations

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Using cached great_expectations-1.23.1-py3-none-any.whl.metadata (10 kB)
INFO: pip is looking at multiple versions of great-expectations to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.1 MB/s eta 0:00:00
ERROR: Cannot install great-expectations==1.23.1, numpy==1.26.4 and pandas==2.2.3 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested numpy==1.26.4
    pandas 2.2.3 depends on numpy>=1.26.0; python_version >= "3.12"
    great-expectations 1.23.1 depends on numpy>=1.22.4; python_version >= "3.10"
    great-expectations 1.23.1 depends on numpy>=1.26.0; python_version >= "3.12"
    great-expectations 1.23.1 depends on numpy>=2.1.0; python_version >= "3.13"

To fix this you could try to:
1. loosen t

Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306.

In [ ]:
from pathlib import Path import pandas as pd

import great_expectations as gx
from great_expectations import expectations as gxe

from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef


datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages



Deprecated call to `pkg_resources.declare_namespace('sphinxcontrib')`.
Implementing implicit namespace packages (as s

### Data location

This notebook was developed in Google Colab. Update `project_dir` to the location of `bank-additional-full.csv` in your own environment.

In [ ]:
# Colab configuration - update this path if the project is stored elsewhere,
project_dir = Path('/content/drive/My Drive/AITesting_Proj')
data_path = project_dir/"bank-additional-full.csv"

bank_data = pd.read_csv(data_path, sep = ';')

context = gx.get_context(mode = 'ephemeral')

# Connect to the data (Give the source a string name)
bank_data_source = context.data_sources.add_pandas(name="bank_data_source")

# Create the Data Asset within that source
bank_data_asset = bank_data_source.add_dataframe_asset(name="bank_data_asset")

# Create suite to store expectations
bank_suite = context.suites.add(gx.ExpectationSuite(name = "bank_suite"))

# Create batch definition for the data asset
batch_definition = bank_data_asset.add_batch_definition_whole_dataframe(name = 'bank_data_batch')
batch = batch_definition.get_batch(batch_parameters = {"dataframe": bank_data})


datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp4xurd0vv' for ephemeral docs site


AttributeError: 'EphemeralDataContext' object has no attribute 'data_sources'

In [ ]:
expectations_container = []

expected_columns = [
    'age', 'job', 'marital', 'education', 'default',
    'housing', 'loan', 'contact', 'month', 'day_of_week',
    'duration', 'campaign', 'pdays', 'previous', 'poutcome',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
    'euribor3m', 'nr.employed', 'y'
]
# Checks the schema without requiring a particular order
expectations_container.append(gxe.ExpectTableColumnsToMatchSet(column_set = expected_columns, exact_match=True))

# Checking if nulls exist in each column
for col in expected_columns:
  expectations_container.append(gxe.ExpectColumnValuesToNotBeNull(column = col, severity = 'warning'))

allowed_categories = {
    "job": ["admin.", "blue-collar", "entrepreneur", "housemaid",
            "management", "retired", "self-employed", "services",
            "student", "technician", "unemployed", "unknown"],
    "marital": ["divorced","married","single","unknown"],
    "education": ["basic.4y", "basic.6y", "basic.9y", "high.school", "illiterate",
                  "professional.course", "university.degree", "unknown"],
    "default": ["no", "yes", "unknown"],
    "housing": ["no", "yes", "unknown"],
    "loan": ["no", "yes", "unknown"],
    "contact": ["cellular", "telephone"],
    "month": ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"],
    "day_of_week": ["mon","tue","wed","thu","fri"],
    'poutcome': ['failure', 'nonexistent', 'success'],
    'y': ['no', 'yes']
}

for col, col_values in allowed_categories.items():
  expectations_container.append(gxe.ExpectColumnValuesToBeOfType(column = col, type_ = 'str'))
  expectations_container.append(gxe.ExpectColumnValuesToBeInSet(column = col, value_set = col_values))

# Expected numerical types when this CSV is loaded by pandas.
integer_cols = ['age', 'duration', 'campaign', 'pdays', 'previous']
float_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in integer_cols:
  expectations_container.append(gxe.ExpectColumnValuesToBeOfType(column = col, type_ = 'int64'))
for col in float_cols:
  expectations_container.append(gxe.ExpectColumnValuesToBeOfType(column = col, type_ = 'float64'))

# Context-related checks for certain numerical columns
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'age', min_value = 1))
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'duration', min_value = 0))
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'campaign', min_value = 1))
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'pdays', min_value = 0, max_value = 999))
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'previous', min_value = 0))
expectations_container.append(gxe.ExpectColumnValuesToBeBetween(column = 'nr.employed', min_value = 0))

for exp in expectations_container:
  bank_suite.add_expectation(exp)

result = batch.validate(bank_suite)
print(result.statistics)

In [ ]:
print("Expectations in suite:", len(bank_suite.expectations))
print("Expectations evaluated:", result.statistics['evaluated_expectations'])

for expectation in bank_suite.expectations:
    print(
        expectation.__class__.__name__,
        "— column:", getattr(expectation, "column", None)
    )